# ML-10 — Content Action Playbook

**Author:** Mehak Zahra  
**Lane:** Refresh / Content Opportunity Scoring  
**Status:** Practical research prototype; not a production automation system

The validated client-holdout result showed that the frozen rule baseline remained the Precision@50 winner (0.78 versus 0.74 for Logistic Regression). This playbook therefore uses that frozen score for review order and transparent rules for action routing. A high score means “review earlier,” never “publish this change automatically.”

## 1. Ranked actions and reason codes

One row is one pseudonymized page from the seven-client held-out validation set. The score uses only visibility, valid position, and CTR opportunity. Archetypes translate measurable conditions into review routes; they do not diagnose why performance changed.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]
            if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists())
df = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
df['is_declining_label'] = df['trend_direction'].eq('down').astype(int)
indices = np.arange(len(df)); SEED = 42
_, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=.20, random_state=SEED)
                   .split(indices, df['is_declining_label'], groups=df['client_id']))
queue = df.iloc[test_idx].copy()
visibility = np.log1p(df['impressions_90d']).rank(method='average', pct=True)
position_opportunity = (((20-df['avg_position'])/19).clip(0,1)
                        * df['avg_position'].between(.000001,20).astype(int))
ctr_gap = ((.50-df['ctr'])/.50).clip(0,1)
all_scores = visibility * position_opportunity * ctr_gap * (df['impressions_90d'] >= 100)
queue['priority_score'] = all_scores.iloc[test_idx].to_numpy()
print(f'Validated queue slice: {len(queue):,} pages across {queue.client_id.nunique()} unseen clients')

Validated queue slice: 6,163 pages across 7 unseen clients


In [2]:
def route(row):
    if row.impressions_90d >= 100 and 0 < row.avg_position <= 20 and row.ctr < .50:
        return 'metadata_opportunity', 'visible_low_ctr', 'review_title_and_snippet'
    if row.impressions_90d >= 1000 and 0 < row.avg_position <= 10 and row.ctr >= .50:
        return 'visible_winner', 'strong_visible_page', 'protect_and_monitor'
    if row.days_since_last_update >= 90 and row.impressions_90d >= 500:
        return 'stale_with_demand', 'stale_but_visible', 'refresh_facts_and_examples'
    if 0 < row.word_count < 1200 and row.impressions_90d >= 250:
        return 'thin_but_visible', 'thin_with_demand', 'review_and_expand_coverage'
    if row.sessions_90d >= 30 and 0 < row.engagement_rate < 30:
        return 'engagement_gap', 'low_engagement', 'review_content_experience'
    return 'weak_or_unclear_signal', 'insufficient_evidence', 'monitor'

routed = queue.apply(route, axis=1, result_type='expand')
routed.columns = ['archetype', 'reason_code', 'action']
queue = pd.concat([queue, routed], axis=1).sort_values(
    ['priority_score', 'impressions_90d'], ascending=False).reset_index(drop=True)
queue['rank'] = np.arange(1, len(queue)+1)
queue['confidence'] = pd.cut(queue['rank'], [0,50,200,np.inf],
 labels=['high_review_priority','medium_review_priority','monitoring_pool']).astype(str)
print(queue.head(10)[['rank','content_id','priority_score','reason_code','action']].to_string(index=False))
print(f"Observed Precision@50: {queue.head(50).is_declining_label.mean():.2f}")

 rank           content_id  priority_score     reason_code                   action
    1 content_d225ec9f3d46        0.861480 visible_low_ctr review_title_and_snippet
    2 content_954cc45bd437        0.825201 visible_low_ctr review_title_and_snippet
    3 content_0022a6b4290f        0.819888 visible_low_ctr review_title_and_snippet
    4 content_929aa622b6a0        0.777423 visible_low_ctr review_title_and_snippet
    5 content_73138e49f25d        0.775295 visible_low_ctr review_title_and_snippet
    6 content_e12868d1f396        0.772452 visible_low_ctr review_title_and_snippet
    7 content_11a4f985f14d        0.766944 visible_low_ctr review_title_and_snippet
    8 content_2d08ccac3f63        0.754335 visible_low_ctr review_title_and_snippet
    9 content_c82bc0c24241        0.752608 visible_low_ctr review_title_and_snippet
   10 content_ceaa28bba4ca        0.730075 visible_low_ctr review_title_and_snippet
Observed Precision@50: 0.78


### Archetype → action mapping

Routing is intentionally conservative. “Refresh,” “expand,” and “review snippet” mean inspect and consider; they are not automatic prescriptions.

In [3]:
archetype_map = pd.DataFrame([
 ['metadata_opportunity','visible_low_ctr','review_title_and_snippet','High visibility + position 1-20 + CTR below 0.50%'],
 ['visible_winner','strong_visible_page','protect_and_monitor','Strong visibility and page-one position'],
 ['stale_with_demand','stale_but_visible','refresh_facts_and_examples','90+ days since update with proven demand'],
 ['thin_but_visible','thin_with_demand','review_and_expand_coverage','Short measured content with existing visibility'],
 ['engagement_gap','low_engagement','review_content_experience','Measured sessions with low engagement'],
 ['weak_or_unclear_signal','insufficient_evidence','monitor','No strong transparent routing signal']],
 columns=['archetype','reason_code','action','rule_evidence'])
print(archetype_map.to_string(index=False))

             archetype           reason_code                     action                                     rule_evidence
  metadata_opportunity       visible_low_ctr   review_title_and_snippet High visibility + position 1-20 + CTR below 0.50%
        visible_winner   strong_visible_page        protect_and_monitor           Strong visibility and page-one position
     stale_with_demand     stale_but_visible refresh_facts_and_examples          90+ days since update with proven demand
      thin_but_visible      thin_with_demand review_and_expand_coverage   Short measured content with existing visibility
        engagement_gap        low_engagement  review_content_experience             Measured sessions with low engagement
weak_or_unclear_signal insufficient_evidence                    monitor              No strong transparent routing signal


### Decay / refresh insight

The Week-4 signal audit found staleness **MIXED**, so age alone is not a refresh order. The paper observed much higher health and impressions among recently refreshed mature pages, but selection effects remain possible. Practical rule: route a stale page to refresh review only when it also has proven demand, then check factual age, seasonality, business value, and historical trajectory before editing.

## 2. Intended use, limits, and cost/value

**User:** a content editor or SEO analyst with a limited review budget. **Use:** open the first 50 rows, inspect real page/query context privately, accept/reject the suggested route, and log the decision. **Boundary:** this 6,163-page held-out slice is a validation artifact based on contemporaneous aggregates; it neither forecasts future Google performance nor estimates causal refresh impact.

The capacity estimate below is an explicit planning assumption—not measured labor cost.

In [4]:
cost_value = pd.DataFrame([
 ['Ranks 1-50',50,20,16.7,'Review first; highest observed queue value'],
 ['Ranks 51-200',150,15,37.5,'Review if capacity remains'],
 ['Ranks 201+',len(queue)-200,0,0.0,'Monitor; no scheduled manual review']],
 columns=['batch','pages','assumed_minutes_each','estimated_hours','decision'])
print(cost_value.to_string(index=False))

       batch  pages  assumed_minutes_each  estimated_hours                                   decision
  Ranks 1-50     50                    20             16.7 Review first; highest observed queue value
Ranks 51-200    150                    15             37.5                 Review if capacity remains
  Ranks 201+   5963                     0              0.0        Monitor; no scheduled manual review


## 3. Human review and no-go list

Before acting, a reviewer must check query intent, SERP features, analytics availability, seasonality, page purpose, factual freshness, cannibalization/duplication, business priority, and whether a recent migration or tracking issue explains the signal.

### Never automate from this score

- Publishing or rewriting page content, titles, or metadata
- Deleting, pruning, redirecting, canonicalizing, or de-indexing a page
- Promising traffic/revenue recovery or claiming Google rewards a feature
- Contacting a client or exposing IDs through a public deliverable
- Applying the score to new clients/data periods without validation
- Treating `monitor` as evidence that a page is safe or unimportant

In [5]:
NO_GO = ['auto_publish','auto_rewrite','auto_delete','auto_prune','auto_redirect',
         'auto_canonical','auto_deindex','promise_recovery','expose_identity']
AUTOMATED_ACTIONS = []
assert not AUTOMATED_ACTIONS
print(f'No-go controls recorded: {len(NO_GO)}; automated content actions: {len(AUTOMATED_ACTIONS)}')

No-go controls recorded: 9; automated content actions: 0


## 4. Monitoring and retrain triggers

Monitoring asks whether ranking quality, inputs, client mix, or editorial usefulness changed. A trigger starts investigation; it does not automatically retrain or deploy.

In [6]:
monitoring = pd.DataFrame([
 ['Precision@50','monthly when proxy matures','below 0.65 twice','pause queue and revalidate'],
 ['Input missingness','each scoring run','+10 percentage points','investigate tracking/schema'],
 ['Score distribution','each scoring run','PSI > 0.20','review drift before reuse'],
 ['New-client share','each scoring run','>25% unseen client mix','run fresh grouped validation'],
 ['Editorial acceptance','monthly','below 50%','review reason rules with editors'],
 ['Outcome data','quarterly','enough reviewed outcomes','consider retraining; keep untouched test']],
 columns=['signal','cadence','trigger','response'])
print(monitoring.to_string(index=False))

              signal                    cadence                  trigger                                 response
        Precision@50 monthly when proxy matures         below 0.65 twice               pause queue and revalidate
   Input missingness           each scoring run    +10 percentage points              investigate tracking/schema
  Score distribution           each scoring run               PSI > 0.20                review drift before reuse
    New-client share           each scoring run   >25% unseen client mix             run fresh grouped validation
Editorial acceptance                    monthly                below 50%         review reason rules with editors
        Outcome data                  quarterly enough reviewed outcomes consider retraining; keep untouched test


## 5. Exports for the paper

The queue CSV is regenerated and stays gitignored. The metrics JSON and accessible SVG action-mix figure are commit-worthy receipts for the paper.

In [7]:
queue['human_check'] = np.select([
 queue.action.eq('review_title_and_snippet'), queue.action.eq('protect_and_monitor'),
 queue.action.eq('refresh_facts_and_examples'), queue.action.eq('review_and_expand_coverage'),
 queue.action.eq('review_content_experience')], [
 'Inspect query intent, SERP features, title/snippet, and tracking before editing.',
 'Confirm strategic value and avoid unnecessary change to a strong page.',
 'Check factual age, historic demand, seasonality, and business priority.',
 'Confirm missing coverage and topic scope; do not pad word count.',
 'Check analytics availability, page purpose, UX, and instrumentation.'],
 default='Collect more evidence before taking any content action.')
out = ROOT/'work'/'outputs'; figures = ROOT/'work'/'figures'
out.mkdir(parents=True, exist_ok=True); figures.mkdir(parents=True, exist_ok=True)
export_cols = ['rank','content_id','client_id','priority_score','confidence','archetype',
 'reason_code','action','human_check','impressions_90d','avg_position','ctr',
 'days_since_last_update','word_count','engagement_rate','is_declining_label']
queue[export_cols].to_csv(out/'action_playbook_queue.csv', index=False)
action_mix = queue.action.value_counts().rename_axis('action').reset_index(name='pages')

# Small dependency-free SVG for the public paper.
from html import escape
lines=['<svg xmlns="http://www.w3.org/2000/svg" width="920" height="420" viewBox="0 0 920 420">',
 '<rect width="100%" height="100%" fill="#ffffff"/>',
 '<text x="460" y="30" text-anchor="middle" font-family="Arial" font-size="22" fill="#17211c">Validated queue: action mix</text>']
maximum=max(action_mix.pages.max(),1)
for i,row in action_mix.iterrows():
 y0=60+i*52; bw=560*row.pages/maximum; label=escape(row.action.replace('_',' '))
 lines += [f'<text x="240" y="{y0+24}" text-anchor="end" font-family="Arial" font-size="13">{label}</text>',
           f'<rect x="250" y="{y0}" width="{bw:.1f}" height="36" rx="5" fill="#195f47"/>',
           f'<text x="{258+bw:.1f}" y="{y0+24}" font-family="Arial" font-size="13">{int(row.pages):,}</text>']
lines.append('</svg>'); (figures/'action_mix.svg').write_text('\n'.join(lines)+'\n')
metrics={'assignment':'ML-10','author':'Mehak Zahra','seed':SEED,
 'validated_queue_rows':len(queue),'held_out_clients':queue.client_id.nunique(),
 'operational_score':'Frozen ML-07 rule baseline','base_rate':round(float(queue.is_declining_label.mean()),6),
 'precision_at_50':round(float(queue.head(50).is_declining_label.mean()),6),
 'action_counts':dict(zip(action_mix.action,action_mix.pages.astype(int))),
 'review_capacity_assumption':'50 pages x 20 minutes = 16.7 hours','automated_actions':[]}
(out/'action_playbook_metrics.json').write_text(json.dumps(metrics,indent=2)+'\n')
print('Wrote work/outputs/action_playbook_queue.csv')
print('Wrote work/outputs/action_playbook_metrics.json')
print('Wrote work/figures/action_mix.svg')
print(action_mix.to_string(index=False))

Wrote work/outputs/action_playbook_queue.csv
Wrote work/outputs/action_playbook_metrics.json
Wrote work/figures/action_mix.svg
                    action  pages
                   monitor   3346
  review_title_and_snippet   2278
refresh_facts_and_examples    239
       protect_and_monitor    236
 review_content_experience     64


## 6. Self-check

- [x] Validated output becomes a ranked, reason-coded action queue.
- [x] Archetype→action mapping and decay/refresh insight are included.
- [x] Intended user, valid use, evidence limits, and cost/value assumptions are explicit.
- [x] Human-review checklist and no-go automation cases are explicit.
- [x] Monitoring/retrain triggers have cadence, threshold, and response.
- [x] Queue CSV, metrics JSON, and paper-ready SVG are regenerated by the notebook.
- [x] Claims remain observed, measured, directional, and decision-support only.
- [x] All cells are executed with visible outputs.

## 7. Five-minute demo outline

**0:00-0:40 — Question and FlyRank content problem**  
Open with: “A content team cannot manually inspect 30,000 pages. Which measurable pages should an editor review first, and which review route fits each page?” Clarify that the output supports prioritization; it does not promise recovery.

**0:40-1:35 — Data and method**  
Show the one-row-per-anonymized-page grain and the observed-decline proxy. Explain the frozen low-CTR visibility rule, Logistic Regression, and Random Forest. Point out the seven-client grouped holdout, zero client overlap, and blocked label-derived fields.

**1:35-2:30 — One chart**  
Show `work/figures/action_mix.svg`, then briefly show the Precision@50 comparison: frozen rule 0.78, Logistic Regression 0.74, Random Forest 0.60. Explain that all methods used the same 6,163 held-out pages.

**2:30-3:25 — One honest result**  
Say: “The model did not beat the simple rule at the primary cutoff. Logistic Regression improved Precision@10 to 0.80, but the frozen rule remained better for the planned batch of 50. Complexity did not earn operational use.”

**3:25-4:25 — One recommendation**  
Recommend reviewing the first 50 high-visibility, low-CTR candidates. For each page, privately verify query intent, SERP features, tracking, seasonality, and business context before changing a title or snippet.

**4:25-5:00 — Limits and close**  
Close with the residual window-overlap limitation and the no-go rule: no automatic publishing, pruning, redirecting, or causal claim. Next step: collect reviewer decisions and later outcomes, then validate a non-overlapping future-window label.

## 8. Two shareable cuts

### Short social post

I built a content-review prioritization study on 30,000 anonymized pages from the FlyRank ML Internship dataset. I compared a transparent low-CTR visibility rule with Logistic Regression and Random Forest on the same 6,163-page holdout containing seven entirely unseen client groups. The honest result was that complexity did not win: the frozen rule reached 0.78 Precision@50, versus 0.74 for Logistic Regression and 0.60 for Random Forest. I turned the result into a reason-coded, human-reviewed action playbook—with explicit leakage checks and no automatic content changes. Read the paper: https://mehkzhra.github.io/FlyRank-ML-Internship/

### Employer-facing three-sentence summary

I built an end-to-end content opportunity ranking pipeline that converts anonymized search and engagement signals into a reproducible, reason-coded editorial review queue. Using 30,000 public-safe FlyRank internship pages, I evaluated a frozen rule, Logistic Regression, and Random Forest on a 6,163-page client-group holdout with seven unseen clients and zero group overlap. The simple rule remained the operational winner at 0.78 Precision@50, so I shipped the negative modeling result honestly as a human-in-the-loop decision-support playbook rather than forcing a more complex model into use.